Phase 1: Load + Basic Metrics

In [1]:
import pandas as pd
import numpy as np

# Load performance data
df = pd.read_csv("mock_performance_data.csv")

# Compute CTR
df["ctr"] = df["clicks"] / df["impressions"]

# Define reward function
df["reward"] = (
    1.0 * df["ctr"]
    + 0.5 * df["engagement_rate"]
    - 2.0 * df["uninstall_rate"]
)

print("\n=== Performance Data with Reward ===")
print(df)


=== Performance Data with Reward ===
  template_id segment_id hour_bucket  impressions  clicks  engagement_rate  \
0          T1         S1     evening          500     120             0.40   
1          T2         S1       night          400      40             0.20   
2          T3         S1     morning          300      60             0.30   
3          T1         S2     evening          450      90             0.35   
4          T2         S2       night          380      30             0.18   
5          T3         S2     morning          320      70             0.32   
6          T1         S3     evening          420     100             0.38   
7          T2         S3       night          390      35             0.22   
8          T3         S3     morning          310      75             0.34   

   uninstall_rate       ctr    reward  
0            0.01  0.240000  0.420000  
1            0.05  0.100000  0.100000  
2            0.02  0.200000  0.310000  
3            0.02  0.

Template Intelligence (Duolingo Logic)

In [2]:
# Global baseline CTR
global_ctr = df["ctr"].mean()

print("Global CTR:", global_ctr)

Global CTR: 0.17860796445898314


In [3]:
template_stats = (
    df.groupby("template_id")
    .agg({
        "impressions": "sum",
        "clicks": "sum"
    })
    .reset_index()
)

template_stats["template_ctr"] = (
    template_stats["clicks"] /
    template_stats["impressions"]
)

print("\n=== Template Stats ===")
print(template_stats)


=== Template Stats ===
  template_id  impressions  clicks  template_ctr
0          T1         1370     310      0.226277
1          T2         1170     105      0.089744
2          T3          930     205      0.220430


In [4]:
template_stats["lift"] = (
    template_stats["template_ctr"] -
    global_ctr
)

print("\n=== Template Lift ===")
print(template_stats)


=== Template Lift ===
  template_id  impressions  clicks  template_ctr      lift
0          T1         1370     310      0.226277  0.047669
1          T2         1170     105      0.089744 -0.088864
2          T3          930     205      0.220430  0.041822


In [5]:
K = 50  # smoothing constant

template_stats["adjusted_lift"] = (
    (template_stats["lift"] * template_stats["impressions"] +
     global_ctr * K)
    /
    (template_stats["impressions"] + K)
)

print("\n=== Adjusted Lift (After Shrinkage) ===")
print(template_stats)


=== Adjusted Lift (After Shrinkage) ===
  template_id  impressions  clicks  template_ctr      lift  adjusted_lift
0          T1         1370     310      0.226277  0.047669       0.052280
1          T2         1170     105      0.089744 -0.088864      -0.077902
2          T3          930     205      0.220430  0.041822       0.048801


Timing Intelligence (Nurture Logic)

In [6]:
# Compute timing score per (segment, hour_bucket)
timing_stats = (
    df.groupby(["segment_id", "hour_bucket"])
    .agg({
        "reward": "mean",
        "impressions": "sum"
    })
    .reset_index()
)

print("\n=== Timing Stats ===")
print(timing_stats)


=== Timing Stats ===
  segment_id hour_bucket    reward  impressions
0         S1     evening  0.420000          500
1         S1     morning  0.310000          300
2         S1       night  0.100000          400
3         S2     evening  0.335000          450
4         S2     morning  0.318750          320
5         S2       night  0.048947          380
6         S3     evening  0.408095          420
7         S3     morning  0.371935          310
8         S3       night  0.099744          390


In [7]:
global_reward = df["reward"].mean()

K_time = 50

timing_stats["adjusted_timing_score"] = (
    (timing_stats["reward"] * timing_stats["impressions"] +
     global_reward * K_time)
    /
    (timing_stats["impressions"] + K_time)
)

print("\n=== Adjusted Timing Score ===")
print(timing_stats)


=== Adjusted Timing Score ===
  segment_id hour_bucket    reward  impressions  adjusted_timing_score
0         S1     evening  0.420000          500               0.406187
1         S1     morning  0.310000          300               0.304007
2         S1       night  0.100000          400               0.118672
3         S2     evening  0.335000          450               0.328305
4         S2     morning  0.318750          320               0.311899
5         S2       night  0.048947          380               0.074425
6         S3     evening  0.408095          420               0.393197
7         S3     morning  0.371935          310               0.357507
8         S3       night  0.099744          390               0.118870


In [10]:
SEND_THRESHOLD = global_reward

timing_stats["allow_send"] = (
    timing_stats["adjusted_timing_score"] > SEND_THRESHOLD
)

print("\n=== Send Decision Per State (Dynamic Threshold) ===")
print(timing_stats)


=== Send Decision Per State (Dynamic Threshold) ===
  segment_id hour_bucket    reward  impressions  adjusted_timing_score  \
0         S1     evening  0.420000          500               0.406187   
1         S1     morning  0.310000          300               0.304007   
2         S1       night  0.100000          400               0.118672   
3         S2     evening  0.335000          450               0.328305   
4         S2     morning  0.318750          320               0.311899   
5         S2       night  0.048947          380               0.074425   
6         S3     evening  0.408095          420               0.393197   
7         S3     morning  0.371935          310               0.357507   
8         S3       night  0.099744          390               0.118870   

   allow_send  
0        True  
1        True  
2       False  
3        True  
4        True  
5       False  
6        True  
7        True  
8       False  


Final Decision Engine

In [11]:
# Create template score dictionary
template_score_dict = dict(
    zip(
        template_stats["template_id"],
        template_stats["adjusted_lift"]
    )
)

print("\n=== Template Score Dictionary ===")
print(template_score_dict)


=== Template Score Dictionary ===
{'T1': 0.05227992036207201, 'T2': -0.07790239360168943, 'T3': 0.048801011506219225}


In [12]:
# Create timing decision dictionary
timing_dict = {}

for _, row in timing_stats.iterrows():
    key = (row["segment_id"], row["hour_bucket"])
    timing_dict[key] = row["allow_send"]

print("\n=== Timing Decision Dictionary ===")
print(timing_dict)


=== Timing Decision Dictionary ===
{('S1', 'evening'): True, ('S1', 'morning'): True, ('S1', 'night'): False, ('S2', 'evening'): True, ('S2', 'morning'): True, ('S2', 'night'): False, ('S3', 'evening'): True, ('S3', 'morning'): True, ('S3', 'night'): False}


In [88]:
def softmax_selection(score_dict, temperature=0.1):
    templates = list(score_dict.keys())
    scores = np.array(list(score_dict.values()))

    # Normalize scores (important)
    scores = scores - np.max(scores)

    exp_scores = np.exp(scores / temperature)
    probabilities = exp_scores / np.sum(exp_scores)

    selected_template = np.random.choice(
        templates,
        p=probabilities
    )

    return selected_template, probabilities

In [109]:
# Simulate user state
user_segment = "S1"
user_hour = "evening"

print("\n=== Simulating Decision ===")

# Step 1: Check timing
if timing_dict[(user_segment, user_hour)]:
    print("Timing allowed. Selecting template...")

    selected_template, probs = softmax_selection(template_score_dict, temperature=0.1)

    print("Selected Template:", selected_template)
    print("Selection Probabilities:", probs)

else:
    print("Timing blocked. No notification sent.")


=== Simulating Decision ===
Timing allowed. Selecting template...
Selected Template: T3
Selection Probabilities: [0.44685858 0.12156134 0.43158008]


Recency Penalty (Recovering Logic)